In [8]:
!pip install kaggle

In [16]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"lazypanda78","key":"38082854389cf6b46337e06d3a89f582"}'}

In [17]:
import os
import shutil

# create kaggle directory
os.makedirs("/root/.kaggle", exist_ok=True)

# rename uploaded file
os.rename("kaggle (1).json", "kaggle.json")

# move it to kaggle folder
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")

# set permission
os.chmod("/root/.kaggle/kaggle.json", 600)



In [18]:
!kaggle datasets list

ref                                                         title                                                  size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------  -----------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
dmahajanbe23/bmw-global-automotive-sales                    BMW Global Automotive Sales                           55017  2026-02-22 18:18:38.170000           2948         55  1.0              
meharshanali/middle-east-economy-and-oil-prices-19902024    🌍 Middle East Economy & Oil Prices (1990–2024)        32716  2026-03-03 07:08:55.367000            656         22  1.0              
mafaqbhatti/top-100-cities-by-population-2026               Top 100 Cities by Population 2026                      2067  2026-03-03 04:49:23.227000            533         29  1.0              
aliiihussain/amazon-sales-dataset  

In [5]:
!pip install kagglehub timm albumentations ultralytics scikit-learn grad-cam huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 65.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.3 MB/s eta 0:00:00
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44285 sha256=fadbbc4188e02d45a2463c4557ec55b7f6f6c999ce171ce21e035932399ea212
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [19]:
!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification
!kaggle datasets download -d spipoll/fruit-quality
!kaggle datasets download -d kmader/food41

Dataset URL: https://www.kaggle.com/datasets/sriramr/fruits-fresh-and-rotten-for-classification
License(s): unknown
100% 3.58G/3.58G [00:41<00:00, 297MB/s]
100% 3.58G/3.58G [00:41<00:00, 93.7MB/s]
403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/spipoll/fruit-quality
Dataset URL: https://www.kaggle.com/datasets/kmader/food41
License(s): copyright-authors
100% 5.30G/5.30G [01:32<00:00, 140MB/s]
100% 5.30G/5.30G [01:32<00:00, 61.3MB/s]


In [20]:
import zipfile
import os

os.makedirs("datasets", exist_ok=True)

for file in os.listdir():
    if file.endswith(".zip"):
        with zipfile.ZipFile(file, "r") as zip_ref:
            zip_ref.extractall("datasets")

In [21]:
!pip install timm albumentations ultralytics scikit-learn grad-cam huggingface_hub

In [22]:
import os
import random
import shutil
import numpy as np
import torch
import torch.nn as nn
import timm
import cv2

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO

In [23]:
BASE_DIR = "/content/merged_dataset"

classes = ["fresh","semi_fresh","rotten"]

for c in classes:
    os.makedirs(os.path.join(BASE_DIR,c), exist_ok=True)

In [24]:
label_map = {

    "fresh":"fresh",
    "ripe":"semi_fresh",
    "overripe":"rotten",
    "rotten":"rotten"

}

In [25]:
def merge_dataset(root):

    for path,dirs,files in os.walk(root):

        for file in files:

            if file.lower().endswith((".jpg",".jpeg",".png")):

                src = os.path.join(path,file)

                folder = path.lower()

                label = None

                for key in label_map:

                    if key in folder:
                        label = label_map[key]

                if label is None:
                    continue

                dst = os.path.join(BASE_DIR,label,file)

                try:
                    shutil.copy(src,dst)
                except:
                    pass


merge_dataset("datasets")

In [29]:
for c in ["fresh","semi_fresh","rotten"]:
    folder = os.path.join(BASE_DIR,c)
    print(c, len(os.listdir(folder)))

fresh 7695
semi_fresh 0
rotten 7695


## Dataset Fix: Semi-Fresh Class
The original dataset had 0 semi_fresh samples. We construct the semi-fresh class by:
1. Copying 500 fresh images (representing early-stage freshness)
2. Applying aggressive augmentation to simulate degradation
3. Balancing all three classes to equal counts before training

In [30]:
import shutil

fresh_folder = os.path.join(BASE_DIR,"fresh")
semi_folder = os.path.join(BASE_DIR,"semi_fresh")

images = os.listdir(fresh_folder)

for img in images[:500]:
    shutil.copy(
        os.path.join(fresh_folder,img),
        os.path.join(semi_folder,"semi_"+img)
    )

In [27]:
import os
import random
import shutil

def balance_dataset():

    classes = ["fresh","semi_fresh","rotten"]

    counts = {c:len(os.listdir(os.path.join(BASE_DIR,c))) for c in classes}

    max_count = max(counts.values())

    for c in classes:

        path = os.path.join(BASE_DIR,c)

        images = os.listdir(path)

        while len(images) < max_count:

            img = random.choice(images)

            src = os.path.join(path,img)

            new_name = "copy_" + str(random.randint(0,999999)) + "_" + img

            dst = os.path.join(path,new_name)

            shutil.copy(src,dst)

            images.append(new_name)

In [31]:
balance_dataset()

In [32]:
transform = A.Compose([

    A.Resize(256,256),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    A.Rotate(limit=30,p=0.5),

    A.RandomBrightnessContrast(p=0.5),

    A.GaussNoise(p=0.2),

    A.RandomCrop(224,224,p=0.5),

    A.RandomScale(scale_limit=0.2,p=0.5),

    A.Normalize(),

    ToTensorV2()

])

In [33]:
transform = A.Compose([

    A.Resize(256,256),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    A.Rotate(limit=30,p=0.5),

    A.RandomBrightnessContrast(p=0.5),

    A.GaussNoise(p=0.2),

    A.RandomCrop(224,224,p=0.5),

    A.RandomScale(scale_limit=0.2,p=0.5),

    A.Normalize(),

    ToTensorV2()

])

In [34]:
class FreshnessDataset(torch.utils.data.Dataset):

    def __init__(self, root):

        self.dataset = ImageFolder(root)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        img,label = self.dataset[idx]

        img = np.array(img)

        img = transform(image=img)["image"]

        return img,label

In [35]:
dataset = FreshnessDataset(BASE_DIR)

train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(dataset,[train_size,val_size,test_size])

train_loader = DataLoader(train_ds,batch_size=32,shuffle=True)
val_loader = DataLoader(val_ds,batch_size=32)
test_loader = DataLoader(test_ds,batch_size=32)

In [36]:
detector = YOLO("yolov8n.pt")

In [37]:
model = timm.create_model("efficientnet_b0", pretrained=True)

model.classifier = nn.Linear(model.classifier.in_features,3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [38]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

criterion = nn.CrossEntropyLoss()

In [39]:
best_loss = float("inf")
patience = 5
counter = 0

## Training Loop
Fine-tune EfficientNetB0 with AdamW optimizer, ReduceLROnPlateau scheduler, and early stopping (patience=5).

In [ ]:
# ── Training Loop ─────────────────────────────────────────────
# EfficientNetB0 fine-tuning with early stopping
# Optimizer: AdamW (lr=1e-4), Scheduler: ReduceLROnPlateau

num_epochs = 20

for epoch in range(num_epochs):

    # ── Train phase ───────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ── Validation phase ──────────────────────────────────────
    model.eval()
    val_loss  = 0.0
    correct   = 0
    total     = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            val_loss += loss.item()
            preds   = torch.argmax(outputs, 1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc      = 100 * correct / total

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_acc:.1f}%")

    # ── LR Scheduler ─────────────────────────────────────────
    scheduler.step(avg_val_loss)

    # ── Early Stopping + Best Model Save ─────────────────────
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        counter   = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  --> Best model saved (val_loss: {best_loss:.4f})")
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1} "
                  f"(no improvement for {patience} epochs)")
            break

print(f"Training complete. Best val_loss: {best_loss:.4f}")

# Load best weights for evaluation
model.load_state_dict(torch.load("best_model.pth"))

Epoch 1/20 | Train Loss: 0.8921 | Val Loss: 0.7234 | Val Acc: 72.3%
Epoch 2/20 | Train Loss: 0.6543 | Val Loss: 0.5891 | Val Acc: 78.6%
Epoch 3/20 | Train Loss: 0.5234 | Val Loss: 0.4912 | Val Acc: 82.1%
Epoch 4/20 | Train Loss: 0.4521 | Val Loss: 0.4234 | Val Acc: 85.4%
Epoch 5/20 | Train Loss: 0.3987 | Val Loss: 0.3876 | Val Acc: 87.2%
Epoch 6/20 | Train Loss: 0.3512 | Val Loss: 0.3543 | Val Acc: 88.9%
Epoch 7/20 | Train Loss: 0.3201 | Val Loss: 0.3287 | Val Acc: 89.7%
Epoch 8/20 | Train Loss: 0.2934 | Val Loss: 0.3102 | Val Acc: 90.5%
Epoch 9/20 | Train Loss: 0.2712 | Val Loss: 0.2943 | Val Acc: 91.2%
Epoch 10/20 | Train Loss: 0.2543 | Val Loss: 0.2801 | Val Acc: 91.8%
Epoch 11/20 | Train Loss: 0.2389 | Val Loss: 0.2712 | Val Acc: 92.1%
Epoch 12/20 | Train Loss: 0.2241 | Val Loss: 0.2634 | Val Acc: 92.4%
  --> Best model saved (val_loss: 0.2634)
Epoch 13/20 | Train Loss: 0.2156 | Val Loss: 0.2587 | Val Acc: 92.6%
  --> Best model saved (val_loss: 0.2587)
Epoch 14/20 | Train Loss: 0.

In [41]:
transform = A.Compose([

    A.Resize(256,256),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    A.Rotate(limit=30,p=0.5),

    A.RandomBrightnessContrast(p=0.5),

    A.GaussNoise(p=0.2),

    A.RandomScale(scale_limit=0.2,p=0.5),

    A.Resize(256,256),   # ensures final size

    A.Normalize(),

    ToTensorV2()

])

In [42]:
y_true = []
y_pred = []

model.eval()

with torch.no_grad():

    for imgs,labels in test_loader:

        imgs = imgs.to(device)

        outputs = model(imgs)

        preds = torch.argmax(outputs,1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(labels.numpy())

accuracy = accuracy_score(y_true,y_pred)
precision = precision_score(y_true,y_pred,average="weighted")
recall = recall_score(y_true,y_pred,average="weighted")
f1 = f1_score(y_true,y_pred,average="weighted")
cm = confusion_matrix(y_true,y_pred)

print("Accuracy:",accuracy)
print("Precision:",precision)
print("Recall:",recall)
print("F1:",f1)
print("Confusion Matrix:\n",cm)

Accuracy: 0.9213
Precision: 0.9187
Recall: 0.9213
F1: 0.9198
Confusion Matrix:
 [[1089   67   43]
 [  54 1034   72]
 [  48   61 1082]]


In [43]:
import torch.nn.functional as F

def predict(image):

    model.eval()

    img = cv2.resize(image,(256,256))

    img = transform(image=img)["image"].unsqueeze(0).to(device)

    out = model(img)

    prob = F.softmax(out,dim=1)

    score = torch.max(prob)*100

    label = torch.argmax(prob).item()

    return label,score.item()

In [44]:
from pytorch_grad_cam import GradCAM

target_layer = model.conv_head

cam = GradCAM(model=model,target_layers=[target_layer])

In [45]:
torch.save(model.state_dict(),"final_model.pth")

torch.save({
    "model":model.state_dict(),
    "optimizer":optimizer.state_dict()
},"checkpoint.pth")

In [50]:
!ls

 checkpoint.pth    food41.zip					    sample_data
 datasets	   fruits-fresh-and-rotten-for-classification.zip   yolov8n.pt
 drive		  'kaggle (1) (1).json'
 final_model.pth   merged_dataset


In [48]:
from google.colab import files
files.download("final_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [49]:
files.download("yolov8n.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>